## Sutskever Encoder-Decoder Architecture
original was on Eng to French while this one is English to Hindi (as i don't know French 😅)

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import os

try:
    # Download the dataset using kagglehub
    path = kagglehub.dataset_download("preetviradiya/english-hindi-dataset")
    print(f"Dataset downloaded to: {path}")

    # List files to see exactly what is inside the directory
    files = []
    for root, dirs, filenames in os.walk(path):
        for f in filenames:
            files.append(os.path.join(root, f))
    print("Files in dataset directory:")
    for f in files:
        print("- ", f)

    # Try to dynamically locate any CSV file if the expected one isn't at the root
    csv_files = [f for f in files if f.endswith('.csv')]
    if csv_files:
        print(f"\nLoading found CSV file: {csv_files[0]}")
        df = pd.read_csv(csv_files[0])
        print("\nFirst 5 records:")
        print(df.head())
    else:
        print("No CSV files found in the dataset directory.")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

Using Colab cache for faster access to the 'english-hindi-dataset' dataset.
Dataset downloaded to: /kaggle/input/english-hindi-dataset
Files in dataset directory:
-  /kaggle/input/english-hindi-dataset/Dataset_English_Hindi.csv

Loading found CSV file: /kaggle/input/english-hindi-dataset/Dataset_English_Hindi.csv

First 5 records:
  English    Hindi
0   Help!    बचाओ!
1   Jump.    उछलो.
2   Jump.    कूदो.
3   Jump.   छलांग.
4  Hello!  नमस्ते।


In [3]:
import re
import time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
# SimpleTokenizer class prepares raw text data for natural language processing (NLP) models by
# mapping words to unique numeric integers.
class SimpleTokenizer:
    FILTERS = '!"#$%&()*+,-./:;<=>?@[\]^_`{|}~\t\n'

    def __init__(self):
        self.word_index = {} # dictionary that will map each unique word to a unique integer index
        self.index_word = {} # reverse dictionary of word_index, mapping integer indices back to their corresponding words
        self._table = str.maketrans({c: " " for c in self.FILTERS}) # maps all punctuation and control characters specified in FILTERS to a space (" ")

    def _split(self, text):
      # It converts the input text to lowercase using .lower().
      # It removes all punctuation by applying the translation table with .translate(self._table).
      # It splits the cleaned text into a list of individual words using .split().
        return text.lower().translate(self._table).split()

    def fit_on_texts(self, texts): # this method builds the vocabulary from a list of documents or strings (texts).
        counts = Counter() #counts the occurrences of each word across all documents using a Counter.
        for t in texts:
            counts.update(self._split(t))
        # then populates self.word_index and self.index_word by sorting words based on their frequency
        self.word_index = {w: i + 1 for i, (w, _) in enumerate(counts.most_common())}
        self.index_word = {i: w for w, i in self.word_index.items()}

    def texts_to_sequences(self, texts): # This method converts the raw strings in texts into lists of integers (sequences).
        # It cleans and splits each text, checks if each word exists in self.word_index, and replaces known words with their corresponding integer index.
        return [[self.word_index[w] for w in self._split(t) if w in self.word_index] for t in texts]


<>:4: SyntaxWarning: invalid escape sequence '\]'
<>:4: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipykernel_448/1571781297.py:4: SyntaxWarning: invalid escape sequence '\]'
  FILTERS = '!"#$%&()*+,-./:;<=>?@[\]^_`{|}~\t\n'


In [5]:
# Initialize tokenizers for English and Hindi separately
english_tokenizer = SimpleTokenizer()
hindi_tokenizer = SimpleTokenizer()

# Convert columns to string, drop NaN values, and fit the tokenizers
english_sentences = df['English'].astype(str).dropna().tolist()
hindi_sentences = df['Hindi'].astype(str).dropna().tolist()

english_tokenizer.fit_on_texts(english_sentences)
hindi_tokenizer.fit_on_texts(hindi_sentences)

print(f"English vocabulary size: {len(english_tokenizer.word_index)}")
print(f"Hindi vocabulary size: {len(hindi_tokenizer.word_index)}")

English vocabulary size: 66117
Hindi vocabulary size: 78537


In [6]:
eng_sequences = english_tokenizer.texts_to_sequences(english_sentences)
hindi_sequences = hindi_tokenizer.texts_to_sequences(hindi_sentences)

# FIX for CUDA out-of-memory: the longest pairs in the dataset are ~400 tokens, and
# padding every batch to that length makes the decoder output tensor (batch x length x
# 78k Hindi words) several GB. Keep only short pairs instead.
MAX_LEN = 30
pairs = [(e, h) for e, h in zip(eng_sequences, hindi_sequences)
         if 0 < len(e) <= MAX_LEN and 0 < len(h) <= MAX_LEN]
eng_sequences, hindi_sequences = map(list, zip(*pairs))
print(f"Kept {len(pairs)} of {len(english_sentences)} sentence pairs (max {MAX_LEN} tokens)")

# SPEED: a 78k-word output layer is the slowest part of training. Keep the 20k most
# frequent Hindi words and map the rest to <unk>. Set HINDI_VOCAB = len(hindi_tokenizer.word_index)
# to turn this off.
HINDI_VOCAB = 20000
UNK_IDX = HINDI_VOCAB + 1
hindi_sequences = [[w if w <= HINDI_VOCAB else UNK_IDX for w in seq] for seq in hindi_sequences]


Kept 111935 of 130476 sentence pairs (max 30 tokens)


 reverse the English source sentences and wrap the Hindi target sentences with <sos> (Start of Sequence) and <eos> (End of Sequence) tokens.

In [7]:
import torch

PAD_IDX = 0
SOS_IDX = UNK_IDX + 1
EOS_IDX = UNK_IDX + 2

# English sequences stay in original order here; they are reversed (and left-padded)
# right after padding in the next cells.
src_tensors = [torch.tensor(seq, dtype=torch.long) for seq in eng_sequences]

# Add <sos> and <eos> to Hindi sequences
trg_tensors = [
    torch.tensor([SOS_IDX] + seq + [EOS_IDX], dtype=torch.long)
    for seq in hindi_sequences
]


In [8]:
trg_tensors[:5]


[tensor([20002, 20001, 20003]),
 tensor([20002, 20001, 20003]),
 tensor([20002, 20001, 20003]),
 tensor([20002, 13689, 20003]),
 tensor([20002, 20001, 20003])]

 Use PyTorch's pad_sequence with batch_first=True so every sentence in your dataset has the same length

In [9]:
from torch.nn.utils.rnn import pad_sequence

# Pad on the right, then flip: gives the REVERSED sentence with padding on the LEFT,
# so the real words sit right next to the decoder instead of behind a run of pad steps.
src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=PAD_IDX).flip(1)
trg_padded = pad_sequence(trg_tensors, batch_first=True, padding_value=PAD_IDX)

print("Padded English shape:", src_padded.shape)
print("Padded Hindi shape:", trg_padded.shape)


Padded English shape: torch.Size([111935, 30])
Padded Hindi shape: torch.Size([111935, 32])


Split Data and Create DataLoaders

In [10]:
from torch.utils.data import DataLoader, TensorDataset, random_split

dataset = TensorDataset(src_padded, trg_padded)

# 80% Train, 20% Validation split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, pin_memory=True, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, pin_memory=True, shuffle=False)


In [11]:
import random
import torch
import torch.nn as nn


class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            emb_dim, hidden_dim, num_layers, dropout=dropout, batch_first=True
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers=2, dropout=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            emb_dim, hidden_dim, num_layers, dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell):
        input_token = input_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_token))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        # Tensor to store all decoder predictions
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(
            self.device
        )

        # 1. Encode the English sentence into context vectors (hidden, cell)
        hidden, cell = self.encoder(src)

        # 2. First input to the decoder is always the <sos> token
        input_token = trg[:, 0]

        # 3. Decode one word at a time
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, t, :] = output

            # Decide whether to use actual next word or model's predicted word
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_token = trg[:, t] if teacher_force else top1

        return outputs


In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INPUT_DIM = len(english_tokenizer.word_index) + 1
OUTPUT_DIM = UNK_IDX + 3
EMB_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.5

enc = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)

model = Seq2Seq(enc, dec, device).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


In [13]:
from tqdm.auto import tqdm

EPOCHS = 15
scaler = torch.amp.GradScaler("cuda")   # mixed precision: faster on T4, less memory

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    start = time.time()

    for src, trg in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        src, trg = src.to(device, non_blocking=True), trg.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            output = model(src, trg)
            # Skip index 0 (<sos>) when computing loss
            output = output[:, 1:, :].reshape(-1, output.shape[-1])
            loss = criterion(output, trg[:, 1:].reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        # Clip gradients to stabilize LSTM training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    print(
        f"Epoch {epoch+1:02}/{EPOCHS} | Train Loss: {epoch_loss / len(train_loader):.4f} "
        f"| {time.time() - start:.0f}s"
    )


Epoch 1/15:   0%|          | 0/700 [00:00<?, ?it/s]

Epoch 01/15 | Train Loss: 6.6013 | 334s


Epoch 2/15:   0%|          | 0/700 [00:00<?, ?it/s]

Epoch 02/15 | Train Loss: 6.1214 | 332s


Epoch 3/15:   0%|          | 0/700 [00:00<?, ?it/s]

Epoch 03/15 | Train Loss: 5.8748 | 332s


... (20 outputs left out)

Epoch 14/15:   0%|          | 0/700 [00:00<?, ?it/s]

Epoch 14/15 | Train Loss: 4.4121 | 332s


Epoch 15/15:   0%|          | 0/700 [00:00<?, ?it/s]

Epoch 15/15 | Train Loss: 4.3337 | 337s


In [14]:
torch.save(model.state_dict(), "sutskever_eng_to_hindi.pt")
print("Model saved successfully!")


Model saved successfully!


In [15]:
# Create a reverse mapping from integer ID -> Hindi word
idx_to_hindi = {idx: word for word, idx in hindi_tokenizer.word_index.items()}


def translate_sentence(sentence, model, device, max_len=50):
    model.eval()

    # 1. Tokenize and reverse the English sentence
    tokens = english_tokenizer.texts_to_sequences([sentence])[0]
    reversed_tokens = tokens[::-1]
    src_tensor = torch.tensor(reversed_tokens, dtype=torch.long).unsqueeze(0).to(device)  # [1, src_len]

    # 2. Encode the source sentence
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    # 3. Start decoding with <sos>
    trg_indexes = [SOS_IDX]

    for _ in range(max_len):
        trg_tensor = torch.tensor([trg_indexes[-1]], dtype=torch.long).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        # Stop if <eos> is predicted
        if pred_token == EOS_IDX:
            break

    # 4. Convert token IDs back to Hindi words (excluding <sos> and <eos>)
    translated_words = [
        idx_to_hindi.get(idx, "<unk>")
        for idx in trg_indexes[1:]
        if idx not in (SOS_IDX, EOS_IDX, PAD_IDX)
    ]
    return " ".join(translated_words)


In [16]:
test_sentences = ["Help!", "How are you?", "I am going home."]

for text in test_sentences:
    translation = translate_sentence(text, model, device)
    print(f"English: {text}")
    print(f"Hindi:   {translation}\n")


English: Help!
Hindi:   की अनुमति दी गई

English: How are you?
Hindi:   निवृत्त हैं

English: I am going home.
Hindi:   मैं तुम्हारे लिए निवृत्त



In [17]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import torch

# Reverse lookup for Hindi IDs -> Words
idx_to_hindi = {idx: word for word, idx in hindi_tokenizer.word_index.items()}


def translate_tensor(src_tensor, model, device, max_len=50):
    """Translates an already tokenized & reversed source tensor into a list of Hindi words."""
    model.eval()
    src_tensor = src_tensor.unsqueeze(0).to(device)  # [1, src_len]

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    trg_indexes = [SOS_IDX]

    for _ in range(max_len):
        trg_tensor = torch.tensor([trg_indexes[-1]], dtype=torch.long).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == EOS_IDX:
            break

    # Convert IDs to list of words (skipping special tokens)
    return [
        idx_to_hindi.get(idx, "<unk>")
        for idx in trg_indexes[1:]
        if idx not in (SOS_IDX, EOS_IDX, PAD_IDX)
    ]


def calculate_bleu(val_dataset, model, device, num_samples=500):
    references = []  # Ground truth sentences
    hypotheses = []  # Model predicted sentences

    # Evaluate on a subset (e.g., 500 samples) for speed, or remove [:num_samples] for full set
    eval_size = min(num_samples, len(val_dataset))

    for i in range(eval_size):
        src_tensor, trg_tensor = val_dataset[i]

        # 1. Get model prediction (list of words)
        pred_words = translate_tensor(src_tensor, model, device)
        hypotheses.append(pred_words)

        # 2. Convert ground-truth target tensor into list of words
        actual_words = [
            idx_to_hindi.get(idx.item(), "<unk>")
            for idx in trg_tensor
            if idx.item() not in (SOS_IDX, EOS_IDX, PAD_IDX)
        ]
        # corpus_bleu expects a list of reference lists per sample: [[ref_words]]
        references.append([actual_words])

    # Smoothing handles short Hindi sentences where 4-gram overlaps might be 0
    smoothie = SmoothingFunction().method4
    score = corpus_bleu(references, hypotheses, smoothing_function=smoothie)

    return score * 100  # Convert from 0-1 scale to 0-100 percentage


bleu_score = calculate_bleu(val_dataset, model, device, num_samples=500)
print(f"Validation BLEU Score: {bleu_score:.2f}")


Validation BLEU Score: 2.39
